# Session V: File I/O and Structured Data

Persist data using text files, JSON, and CSV, and learn how to design file-based mini databases.

## Exercise 1: Daily Journal App
**Objective:** Practice reading, writing, and appending to text files.

Build a simple daily journal application that stores entries in a plain text file (`journal.txt`). Each entry should include a timestamp and the user's text.

**Tasks:**
1. `add_entry(filename, text)` → Append a new journal entry with the current date/time to the file.
2. `read_journal(filename)` → Read and display all entries from the file. Handle the case where the file doesn't exist yet.
3. `search_journal(filename, keyword)` → Read the file and return only lines that contain the keyword (case-insensitive).
4. `count_entries(filename)` → Return the total number of entries in the journal.
5. `run_journal()` → A menu-driven interface: [A]dd entry, [R]ead all, [S]earch, [C]ount, [Q]uit.

In [2]:
from datetime import datetime

JOURNAL_FILE = "journal.txt"

def add_entry(filename, text):
    """Append a timestamped journal entry to the file."""
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    entry = f"[{timestamp}] {text.strip()}\n"

    with open(filename, "a", encoding="utf-8") as file:
        file.write(entry)
    print("Entry added successfully.")

def read_journal(filename):
    """Read and print all entries. Handle FileNotFoundError."""
    try:
        with open(filename, "r", encoding="utf-8") as file:
            entries = [line.strip() for line in file if line.strip()]

        if not entries:
            print("Journal is empty.")
            return

        print("\n--- Journal Entries ---")
        for entry in entries:
            print(entry)
        print("-----------------------")
    except FileNotFoundError:
        print("No journal file found. Add an entry first!")

def search_journal(filename, keyword):
    """Return a list of lines containing the keyword (case-insensitive)."""
    keyword_lower = keyword.lower()
    matches = []

    try:
        with open(filename, "r", encoding="utf-8") as file:
            for line in file:
                cleaned = line.strip()
                if cleaned and keyword_lower in cleaned.lower():
                    matches.append(cleaned)
    except FileNotFoundError:
        print("No journal file found to search.")

    return matches

def count_entries(filename):
    """Count and return the number of journal entries."""
    try:
        with open(filename, "r", encoding="utf-8") as file:
            return sum(1 for line in file if line.strip())
    except FileNotFoundError:
        return 0

def run_journal():
    """Menu-driven journal interface."""
    menu = """
=======================
      MY JOURNAL
=======================
[A]dd entry
[R]ead all entries
[S]earch entries
[C]ount entries
[Q]uit
"""
    while True:
        print(menu)
        choice = input("Choose an option: ").strip().upper()

        if choice == "A":
            entry_text = input("Enter your journal entry: ").strip()
            if entry_text:
                add_entry(JOURNAL_FILE, entry_text)
            else:
                print("Cannot save an empty entry.")

        elif choice == "R":
            read_journal(JOURNAL_FILE)

        elif choice == "S":
            keyword = input("Enter search keyword: ").strip()
            if keyword:
                results = search_journal(JOURNAL_FILE, keyword)
                if results:
                    print(f"\nFound {len(results)} matching entry/entries:")
                    for res in results:
                        print(f" - {res}")
                else:
                    print(f"No entries found containing '{keyword}'.")
            else:
                print("Keyword cannot be empty.")

        elif choice == "C":
            total = count_entries(JOURNAL_FILE)
            print(f"\nTotal entries in journal: {total}")

        elif choice == "Q":
            print("Goodbye!")
            break

        else:
            print("Invalid selection. Please choose A, R, S, C, or Q.")


if __name__ == "__main__":
    # --- Quick non-interactive sanity check ---
    add_entry(JOURNAL_FILE, "Started learning about file I/O today!")
    add_entry(JOURNAL_FILE, "Python makes reading files so easy.")
    read_journal(JOURNAL_FILE)
    print("Search results:", search_journal(JOURNAL_FILE, "python"))
    print("Total entries:", count_entries(JOURNAL_FILE))

    run_journal()


Entry added successfully.
Entry added successfully.

--- Journal Entries ---
[2026-10-19 11:09:38] Started learning about file I/O today!
[2026-10-19 11:17:04] Started learning about file I/O today!
[2026-10-19 11:17:04] Python makes reading files so easy.
-----------------------
Search results: ['[2026-10-19 11:17:04] Python makes reading files so easy.']
Total entries: 3

      MY JOURNAL
[A]dd entry
[R]ead all entries
[S]earch entries
[C]ount entries
[Q]uit

Entry added successfully.

      MY JOURNAL
[A]dd entry
[R]ead all entries
[S]earch entries
[C]ount entries
[Q]uit



KeyboardInterrupt: Interrupted by user

---
## Exercise 2: To-Do List Manager (JSON Persistence)
**Objective:** Build a CRUD application that stores data in a JSON file.

Create a to-do list manager where tasks are stored as a list of dictionaries in a `tasks.json` file. Each task has a `title`, a `priority` (high/medium/low), and a `done` status (True/False).

**Tasks:**
1. `load_tasks(filename)` → Load tasks from JSON. Return an empty list if the file is missing or corrupted.
2. `save_tasks(filename, tasks)` → Save the tasks list to JSON with pretty formatting.
3. `add_task(tasks)` → Prompt for title and priority, append a new task dict (done=False).
4. `list_tasks(tasks)` → Display all tasks in a formatted table with their index, status, title, and priority.
5. `mark_done(tasks)` → Ask for a task index and set its `done` field to True.
6. `delete_task(tasks)` → Ask for a task index and remove the task from the list.
7. `run_todo()` → Menu loop: [A]dd, [L]ist, [D]one, [R]emove, [Q]uit. Auto-save after every change.

**Data structure:**
```python
[
    {"title": "Buy groceries", "priority": "high", "done": False},
    {"title": "Read chapter 5", "priority": "medium", "done": True}
]
```

In [4]:
import json

TODO_FILE = "tasks.json"

def load_tasks(filename):
    """Load tasks from a JSON file. Return [] if missing or corrupted."""
    try:
        with open(filename, "r", encoding="utf-8") as f:
            data = json.load(f)
            return data if isinstance(data, list) else []
    except (FileNotFoundError, json.JSONDecodeError):
        return []

def save_tasks(filename, tasks):
    """Save tasks list to a JSON file with indent=2."""
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(tasks, f, indent=2)

def add_task(tasks):
    """Prompt for title and priority, append new task."""
    while True:
        title = input("Enter task title: ").strip()
        if title:
            break
        print("Title cannot be empty. Please try again.")

    valid_priorities = ("high", "medium", "low")
    while True:
        priority = input("Enter priority (high/medium/low) [medium]: ").strip().lower()
        if not priority:
            priority = "medium"
        if priority in valid_priorities:
            break
        print("Invalid priority. Choose from 'high', 'medium', or 'low'.")

    new_task = {
        "title": title,
        "priority": priority,
        "done": False
    }
    tasks.append(new_task)
    print(f"Task '{title}' added.")

def list_tasks(tasks):
    """Display all tasks in a formatted table."""
    if not tasks:
        print("\nNo tasks to display.")
        return

    print("\n" + "=" * 56)
    print(f"{'#':<4} {'Status':<8} {'Priority':<10} {'Title'}")
    print("-" * 56)
    for idx, task in enumerate(tasks, start=1):
        status = "[x]" if task.get("done") else "[ ]"
        priority = task.get("priority", "medium").capitalize()
        title = task.get("title", "Untitled")
        print(f"{idx:<4} {status:<8} {priority:<10} {title}")
    print("=" * 56)

def _get_valid_index(tasks, prompt_action):
    """Helper to prompt for and validate a 1-based task index."""
    if not tasks:
        print(f"No tasks available to {prompt_action}.")
        return None

    while True:
        raw_val = input(f"Enter task number to {prompt_action} (or press Enter to cancel): ").strip()
        if not raw_val:
            return None
        try:
            task_num = int(raw_val)
            if 1 <= task_num <= len(tasks):
                return task_num - 1  # Convert to 0-based index
            print(f"Please enter a number between 1 and {len(tasks)}.")
        except ValueError:
            print("Invalid input. Please enter a valid whole number.")

def mark_done(tasks):
    """Ask for a task index and mark it as done."""
    idx = _get_valid_index(tasks, "mark as done")
    if idx is not None:
        if tasks[idx]["done"]:
            print(f"Task '{tasks[idx]['title']}' was already marked as done.")
        else:
            tasks[idx]["done"] = True
            print(f"Task '{tasks[idx]['title']}' marked as done.")

def delete_task(tasks):
    """Ask for a task index and remove the task."""
    idx = _get_valid_index(tasks, "remove")
    if idx is not None:
        removed = tasks.pop(idx)
        print(f"Removed task: '{removed['title']}'")

def run_todo():
    """Main menu loop with auto-save."""
    tasks = load_tasks(TODO_FILE)

    menu = """
============================
      TODO LIST MANAGER
============================
[A]dd task
[L]ist tasks
[D]one (mark completed)
[R]emove task
[Q]uit
"""
    while True:
        print(menu)
        choice = input("Choose an option: ").strip().upper()

        if choice == "A":
            add_task(tasks)
            save_tasks(TODO_FILE, tasks)
        elif choice == "L":
            list_tasks(tasks)
        elif choice == "D":
            list_tasks(tasks)
            mark_done(tasks)
            save_tasks(TODO_FILE, tasks)
        elif choice == "R":
            list_tasks(tasks)
            delete_task(tasks)
            save_tasks(TODO_FILE, tasks)
        elif choice == "Q":
            print("Changes saved. Goodbye!")
            break
        else:
            print("Invalid option. Please choose A, L, D, R, or Q.")


if __name__ == "__main__":
    # --- QUICK TEST (non-interactive) ---
    test_tasks = load_tasks(TODO_FILE)
    test_tasks.append({"title": "Test task", "priority": "high", "done": False})
    save_tasks(TODO_FILE, test_tasks)
    reloaded = load_tasks(TODO_FILE)
    list_tasks(reloaded)

    # Uncomment to run the interactive loop:
    run_todo()


#    Status   Priority   Title
--------------------------------------------------------
1    [ ]      High       Test task
2    [ ]      High       Test task

      TODO LIST MANAGER
[A]dd task
[L]ist tasks
[D]one (mark completed)
[R]emove task
[Q]uit

Task 'Clean my room!' added.

      TODO LIST MANAGER
[A]dd task
[L]ist tasks
[D]one (mark completed)
[R]emove task
[Q]uit


#    Status   Priority   Title
--------------------------------------------------------
1    [ ]      High       Test task
2    [ ]      High       Test task
3    [ ]      High       Clean my room!

      TODO LIST MANAGER
[A]dd task
[L]ist tasks
[D]one (mark completed)
[R]emove task
[Q]uit


#    Status   Priority   Title
--------------------------------------------------------
1    [ ]      High       Test task
2    [ ]      High       Test task
3    [ ]      High       Clean my room!
Removed task: 'Test task'

      TODO LIST MANAGER
[A]dd task
[L]ist tasks
[D]one (mark completed)
[R]emove task
[Q]uit

Changes 

---
## Exercise 3: CSV Expense Importer
**Objective:** Read, validate, and analyze data from a CSV file.

You receive a CSV file of monthly expenses. Some rows may have missing or invalid data. Write functions to import the data safely, clean it, and produce summary reports.

**First, run the cell below to generate the sample CSV file.** Then write the functions to process it.

**Tasks:**
1. `load_expenses(filename)` → Read the CSV using `csv.DictReader`. Validate each row: skip rows where `amount` is not a valid number or `category` is empty. Return a list of clean dicts (with `amount` converted to float).
2. `total_by_category(expenses)` → Return a dict mapping each category to its total spending.
3. `most_expensive(expenses)` → Return the single most expensive entry.
4. `print_report(expenses)` → Print a formatted summary: total spent, breakdown by category, and the most expensive item.

In [6]:
import csv

# --- RUN THIS CELL FIRST to generate the sample CSV ---
sample_data = """date,description,category,amount
2024-01-05,Grocery store,Food,52.30
2024-01-06,Bus pass,Transport,40.00
2024-01-07,Coffee shop,Food,4.50
2024-01-08,Netflix,Entertainment,15.99
2024-01-09,Gas station,Transport,bad_data
2024-01-10,Restaurant,Food,38.75
2024-01-11,Gym membership,,45.00
2024-01-12,Bookstore,Education,22.50
2024-01-13,Electricity bill,Utilities,89.00
2024-01-14,Movie tickets,Entertainment,24.00
2024-01-15,Taxi,Transport,18.50
2024-01-16,Online course,Education,49.99
2024-01-17,Supermarket,Food,67.20
2024-01-18,Water bill,Utilities,not_a_number
2024-01-19,Concert,Entertainment,75.00
2024-01-20,Pharmacy,Health,12.40"""

with open("expenses.csv", "w", newline="") as f:
    f.write(sample_data)

print("✅ expenses.csv created successfully.")

✅ expenses.csv created successfully.


In [7]:
import csv

def load_expenses(filename):
    """Load and validate expenses from CSV. Skip invalid rows."""
    clean_expenses = []

    try:
        with open(filename, "r", encoding="utf-8", newline="") as f:
            reader = csv.DictReader(f)
            for row in reader:
                # 1. Validate category is non-empty
                category = row.get("category", "").strip()
                if not category:
                    continue

                # 2. Validate amount is a valid float
                raw_amount = row.get("amount", "").strip()
                try:
                    amount = float(raw_amount)
                except ValueError:
                    continue  # Skip rows with invalid numbers like 'bad_data'

                # Store clean dict with converted amount
                clean_expenses.append({
                    "date": row.get("date", "").strip(),
                    "description": row.get("description", "").strip(),
                    "category": category,
                    "amount": round(amount, 2)
                })
    except FileNotFoundError:
        print(f"File '{filename}' not found.")
        return []

    return clean_expenses

def total_by_category(expenses):
    """Return {category: total_amount} dict."""
    totals = {}
    for entry in expenses:
        category = entry["category"]
        totals[category] = round(totals.get(category, 0.0) + entry["amount"], 2)
    return totals

def most_expensive(expenses):
    """Return the expense dict with the highest amount."""
    if not expenses:
        return None
    return max(expenses, key=lambda x: x["amount"])

def print_report(expenses):
    """Print a formatted expense summary report."""
    if not expenses:
        print("\nNo expense data available to generate a report.")
        return

    total_spent = sum(item["amount"] for item in expenses)
    cat_totals = total_by_category(expenses)
    top_expense = most_expensive(expenses)

    print("\n" + "=" * 42)
    print("           EXPENSE REPORT")
    print("=" * 42)
    print(f"Total Transactions Validated: {len(expenses)}")
    print(f"Total Amount Spent:          ${total_spent:,.2f}")

    print("\n" + "-" * 42)
    print("Spending by Category:")
    print("-" * 42)
    # Sort categories by spending descending
    for category, total in sorted(cat_totals.items(), key=lambda item: item[1], reverse=True):
        percentage = (total / total_spent) * 100 if total_spent else 0
        print(f"  • {category:<15} ${total:>8.2f}  ({percentage:>5.1f}%)")

    print("-" * 42)
    print("Most Expensive Expense:")
    print("-" * 42)
    if top_expense:
        print(f"  Description: {top_expense['description']}")
        print(f"  Category:    {top_expense['category']}")
        print(f"  Date:        {top_expense['date']}")
        print(f"  Amount:      ${top_expense['amount']:,.2f}")
    print("=" * 42)


# --- TEST ---
if __name__ == "__main__":
    expenses = load_expenses("expenses.csv")
    print_report(expenses)


           EXPENSE REPORT
Total Transactions Validated: 13
Total Amount Spent:          $510.13

------------------------------------------
Spending by Category:
------------------------------------------
  • Food            $  162.75  ( 31.9%)
  • Entertainment   $  114.99  ( 22.5%)
  • Utilities       $   89.00  ( 17.4%)
  • Education       $   72.49  ( 14.2%)
  • Transport       $   58.50  ( 11.5%)
  • Health          $   12.40  (  2.4%)
------------------------------------------
Most Expensive Expense:
------------------------------------------
  Description: Electricity bill
  Category:    Utilities
  Date:        2024-01-13
  Amount:      $89.00


---
## Exercise 4: File Content Validator
**Objective:** Write robust validation functions for both JSON and CSV files.

You will build a validation toolkit that checks whether files are well-formed and contain the expected structure *before* your application tries to use the data.

**First, run the cell below to generate sample test files** (some valid, some intentionally broken). Then write the validators.

**Tasks:**
1. `validate_json(filename, required_keys)` → Check that the file (a) exists, (b) contains valid JSON, (c) is a list of dicts, and (d) each dict contains all `required_keys`. Return a report dict: `{"valid": int, "invalid": int, "errors": [str]}`.
2. `validate_csv(filename, expected_columns)` → Check that the file (a) exists, (b) has the correct number of columns in the header, and (c) each data row has the correct number of fields. Return a similar report dict.
3. `print_validation_report(report, filename)` → Pretty-print the validation results.

In [13]:
import json

# --- RUN THIS CELL FIRST to generate test files ---

# Good JSON file
good_json = [
    {"name": "Alice", "age": 22, "email": "alice@test.com"},
    {"name": "Bob", "age": 25, "email": "bob@test.com"},
    {"name": "Charlie", "age": 30, "email": "charlie@test.com"},
]
with open("good_data.json", "w") as f:
    json.dump(good_json, f, indent=2)

# Bad JSON file (missing keys, wrong types)
bad_json = [
    {"name": "Alice", "age": 22, "email": "alice@test.com"},
    {"name": "Bob", "age": 25},
    "this is not a dict",
    {"name": "Diana", "age": "not_a_number", "email": "diana@test.com"},
]
with open("bad_data.json", "w") as f:
    json.dump(bad_json, f, indent=2)

# Corrupted JSON file
with open("corrupted.json", "w") as f:
    f.write('{"name": "broken", missing_quotes: true}')

# Good CSV file
with open("good_data.csv", "w", newline="") as f:
    f.write("name,age,email\n")
    f.write("Alice,22,alice@test.com\n")
    f.write("Bob,25,bob@test.com\n")

# Bad CSV file (inconsistent columns)
with open("bad_data.csv", "w", newline="") as f:
    f.write("name,age,email\n")
    f.write("Alice,22,alice@test.com\n")
    f.write("Bob,25\n")
    f.write("Charlie,30,charlie@test.com,extra_field\n")
    f.write("Diana,28,diana@test.com\n")

print("✅ All test files created successfully.")

✅ All test files created successfully.


In [14]:
import json
import csv

def validate_json(filename, required_keys):
    """Validate a JSON file's structure and required keys.
    Return {"valid": int, "invalid": int, "errors": [str]}."""
    report = {"valid": 0, "invalid": 0, "errors": []}
    required_set = set(required_keys)

    # 1. Existence and syntax parsing
    try:
        with open(filename, "r", encoding="utf-8") as f:
            data = json.load(f)
    except FileNotFoundError:
        report["errors"].append(f"File not found: '{filename}'")
        return report
    except json.JSONDecodeError as e:
        report["errors"].append(f"JSON syntax error: {e.msg} (line {e.lineno}, col {e.colno})")
        return report

    # 2. Check root element is a list
    if not isinstance(data, list):
        report["errors"].append(f"Root JSON structure must be a list, found: {type(data).__name__}")
        return report

    # 3. Check individual items
    for idx, item in enumerate(data):
        if not isinstance(item, dict):
            report["invalid"] += 1
            report["errors"].append(f"Item #{idx}: Expected dict, found {type(item).__name__} ({repr(item)})")
            continue

        missing = [key for key in required_keys if key not in item]
        if missing:
            report["invalid"] += 1
            report["errors"].append(f"Item #{idx}: Missing required key(s): {missing}")
        else:
            report["valid"] += 1

    return report

def validate_csv(filename, expected_columns):
    """Validate a CSV file's structure and column count.
    expected_columns can be an integer (count) or a list of column names.
    Return {"valid": int, "invalid": int, "errors": [str]}."""
    report = {"valid": 0, "invalid": 0, "errors": []}

    target_count = len(expected_columns) if isinstance(expected_columns, (list, tuple)) else int(expected_columns)

    # 1. Open and inspect file
    try:
        with open(filename, "r", encoding="utf-8", newline="") as f:
            reader = csv.reader(f)

            # Read and inspect header
            try:
                header = next(reader)
            except StopIteration:
                report["errors"].append("CSV file is completely empty.")
                return report

            if len(header) != target_count:
                report["errors"].append(
                    f"Header column mismatch: expected {target_count} columns, found {len(header)} {header}"
                )

            # Inspect each data row
            for row_num, row in enumerate(reader, start=2):
                # Skip trailing empty lines
                if not row:
                    continue
                if len(row) != target_count:
                    report["invalid"] += 1
                    report["errors"].append(
                        f"Row {row_num}: Expected {target_count} fields, found {len(row)} ({row})"
                    )
                else:
                    report["valid"] += 1

    except FileNotFoundError:
        report["errors"].append(f"File not found: '{filename}'")

    return report

def print_validation_report(report, filename):
    """Pretty-print the validation results."""
    total_records = report["valid"] + report["invalid"]
    status_symbol = "PASSED" if not report["errors"] and report["valid"] > 0 else "FAILED"

    print(f"Validation Report for: {filename}")
    print(f"Status:       [{status_symbol}]")
    print(f"Valid Rows:   {report['valid']}")
    print(f"Invalid Rows: {report['invalid']}")
    print(f"Total Errors: {len(report['errors'])}")

    if report["errors"]:
        print("\nDetails:")
        for err in report["errors"]:
            print(f"  • {err}")

# --- TEST ---
print("=" * 50)
report = validate_json("good_data.json", ["name", "age", "email"])
print_validation_report(report, "good_data.json")

print("=" * 50)
report = validate_json("bad_data.json", ["name", "age", "email"])
print_validation_report(report, "bad_data.json")

print("=" * 50)
report = validate_json("corrupted.json", ["name", "age", "email"])
print_validation_report(report, "corrupted.json")

print("=" * 50)
report = validate_csv("good_data.csv", 3)
print_validation_report(report, "good_data.csv")

print("=" * 50)
report = validate_csv("bad_data.csv", 3)
print_validation_report(report, "bad_data.csv")

Validation Report for: good_data.json
Status:       [PASSED]
Valid Rows:   3
Invalid Rows: 0
Total Errors: 0
Validation Report for: bad_data.json
Status:       [FAILED]
Valid Rows:   2
Invalid Rows: 2
Total Errors: 2

Details:
  • Item #1: Missing required key(s): ['email']
  • Item #2: Expected dict, found str ('this is not a dict')
Validation Report for: corrupted.json
Status:       [FAILED]
Valid Rows:   0
Invalid Rows: 0
Total Errors: 1

Details:
  • JSON syntax error: Expecting property name enclosed in double quotes (line 1, col 20)
Validation Report for: good_data.csv
Status:       [PASSED]
Valid Rows:   2
Invalid Rows: 0
Total Errors: 0
Validation Report for: bad_data.csv
Status:       [FAILED]
Valid Rows:   2
Invalid Rows: 2
Total Errors: 2

Details:
  • Row 3: Expected 3 fields, found 2 (['Bob', '25'])
  • Row 4: Expected 3 fields, found 4 (['Charlie', '30', 'charlie@test.com', 'extra_field'])


---
## Exercise 5: Format Converter & Exporter
**Objective:** Convert data between JSON and CSV formats in both directions.

Build a bidirectional converter that can transform data files between JSON and CSV. This is an extremely common real-world task — APIs return JSON, but spreadsheets need CSV, and vice versa.

**Tasks:**
1. `json_to_csv(json_filename, csv_filename)` → Read a JSON file (list of dicts) and write it as a CSV file. Use the keys of the first dict as column headers.
2. `csv_to_json(csv_filename, json_filename)` → Read a CSV file and write it as a JSON file (list of dicts).
3. `export_filtered_csv(json_filename, csv_filename, filter_key, filter_value)` → Read a JSON file, keep only records where `record[filter_key] == filter_value`, and export the filtered data to CSV.
4. Handle edge cases: empty files, missing files, and mismatched structures.

In [15]:
import json
import csv

# --- Setup: create a test JSON file ---
students = [
    {"name": "Alice", "age": 22, "city": "Madrid", "grade": "A"},
    {"name": "Bob", "age": 25, "city": "London", "grade": "B"},
    {"name": "Charlie", "age": 30, "city": "Paris", "grade": "A"},
    {"name": "Diana", "age": 28, "city": "Madrid", "grade": "C"},
    {"name": "Eve", "age": 23, "city": "London", "grade": "A"},
]

with open("students.json", "w") as f:
    json.dump(students, f, indent=2)

print("✅ students.json created.")

✅ students.json created.


In [16]:
import json
import csv

def _write_dicts_to_csv(data, csv_filename):
    """Helper to write a list of dicts to CSV using the first item's keys as headers."""
    if not data or not isinstance(data[0], dict):
        print(f"No valid dict records to write to '{csv_filename}'.")
        return False

    fieldnames = list(data[0].keys())
    with open(csv_filename, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(data)
    return True

def json_to_csv(json_filename, csv_filename):
    """Convert a JSON file (list of dicts) to a CSV file."""
    try:
        with open(json_filename, "r", encoding="utf-8") as f:
            data = json.load(f)
    except FileNotFoundError:
        print(f"Error: File '{json_filename}' not found.")
        return False
    except json.JSONDecodeError as e:
        print(f"Error parsing JSON from '{json_filename}': {e}")
        return False

    if not isinstance(data, list):
        print(f"Error: JSON root in '{json_filename}' must be a list of dicts.")
        return False

    if not data:
        print(f"Warning: '{json_filename}' contains an empty list. Nothing written.")
        return False

    if _write_dicts_to_csv(data, csv_filename):
        print(f"Successfully converted '{json_filename}' to '{csv_filename}'.")
        return True
    return False

def csv_to_json(csv_filename, json_filename):
    """Convert a CSV file to a JSON file (list of dicts)."""
    try:
        with open(csv_filename, "r", encoding="utf-8", newline="") as f:
            reader = csv.DictReader(f)
            records = list(reader)
    except FileNotFoundError:
        print(f"Error: File '{csv_filename}' not found.")
        return False

    with open(json_filename, "w", encoding="utf-8") as f:
        json.dump(records, f, indent=2)

    print(f"Successfully converted '{csv_filename}' to '{json_filename}' ({len(records)} records).")
    return True

def export_filtered_csv(json_filename, csv_filename, filter_key, filter_value):
    """Export only records where record[filter_key] == filter_value to CSV."""
    try:
        with open(json_filename, "r", encoding="utf-8") as f:
            data = json.load(f)
    except FileNotFoundError:
        print(f"Error: File '{json_filename}' not found.")
        return False
    except json.JSONDecodeError as e:
        print(f"Error parsing JSON from '{json_filename}': {e}")
        return False

    if not isinstance(data, list):
        print(f"Error: Expected a list in '{json_filename}'.")
        return False

    # Filter matching items safely checking for key existence
    filtered_data = [
        item for item in data
        if isinstance(item, dict) and item.get(filter_key) == filter_value
    ]

    if not filtered_data:
        print(f"No records found where '{filter_key}' == '{filter_value}'. No file created.")
        return False

    if _write_dicts_to_csv(filtered_data, csv_filename):
        print(f"Exported {len(filtered_data)} matching records to '{csv_filename}'.")
        return True
    return False


# --- TEST ---
if __name__ == "__main__":
    # JSON -> CSV
    json_to_csv("students.json", "students_export.csv")

    # CSV -> JSON (round-trip test)
    csv_to_json("students_export.csv", "students_roundtrip.json")

    # Filtered export: only students from Madrid
    export_filtered_csv("students.json", "madrid_students.csv", "city", "Madrid")

    # Verify round-trip
    with open("students_roundtrip.json", "r") as f:
        roundtrip = json.load(f)
    print(f"\nRound-trip data ({len(roundtrip)} records):")
    for r in roundtrip:
        print(f"  {r}")

Successfully converted 'students.json' to 'students_export.csv'.
Successfully converted 'students_export.csv' to 'students_roundtrip.json' (5 records).
Exported 2 matching records to 'madrid_students.csv'.

Round-trip data (5 records):
  {'name': 'Alice', 'age': '22', 'city': 'Madrid', 'grade': 'A'}
  {'name': 'Bob', 'age': '25', 'city': 'London', 'grade': 'B'}
  {'name': 'Charlie', 'age': '30', 'city': 'Paris', 'grade': 'A'}
  {'name': 'Diana', 'age': '28', 'city': 'Madrid', 'grade': 'C'}
  {'name': 'Eve', 'age': '23', 'city': 'London', 'grade': 'A'}


---
## Exercise 6: Student Records Database (Capstone)
**Objective:** Build a complete file-based CRUD application combining JSON persistence, CSV import/export, and data validation.

Create a student records management system that stores data in a JSON file and supports importing/exporting CSV. This exercise combines **every concept** from this session.

**Required features:**
1. **[A]dd Student** — Prompt for name, age, email, and grade. Validate all inputs (age must be a positive integer, email must contain `@`, grade must be A/B/C/D/F). Store in JSON.
2. **[L]ist Students** — Display all students in a formatted table.
3. **[S]earch** — Find a student by name (case-insensitive partial match).
4. **[U]pdate** — Modify a student's grade or email by name.
5. **[D]elete** — Remove a student by name.
6. **[I]mport CSV** — Load students from a CSV file, validating each row before adding. Skip invalid rows and report what was skipped.
7. **[E]xport CSV** — Export all current students to a CSV file.
8. **[Q]uit** — Exit the program.

**Data structure (JSON):**
```python
[
    {"name": "Alice", "age": 22, "email": "alice@uni.edu", "grade": "A"},
    {"name": "Bob", "age": 25, "email": "bob@uni.edu", "grade": "B"}
]
```

In [19]:
import json
import csv

DB_FILE = "students_db.json"
VALID_GRADES = {"A", "B", "C", "D", "F"}

# --- PERSISTENCE ---

def load_db(filename):
    """Load student records from JSON. Return [] if missing/corrupted."""
    try:
        with open(filename, "r", encoding="utf-8") as f:
            data = json.load(f)
            return data if isinstance(data, list) else []
    except (FileNotFoundError, json.JSONDecodeError):
        return []

def save_db(filename, students):
    """Save student records to JSON."""
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(students, f, indent=2)

# --- VALIDATION ---

def validate_student(name, age, email, grade):
    """Validate student fields. Return (is_valid, parsed_data_or_error_message)."""
    # 1. Name validation
    cleaned_name = str(name).strip()
    if not cleaned_name:
        return False, "Name cannot be blank."

    # 2. Age validation (must resolve to positive integer)
    try:
        age_int = int(str(age).strip())
        if age_int <= 0:
            return False, "Age must be a positive integer greater than zero."
    except ValueError:
        return False, f"Age '{age}' is not a valid integer."

    # 3. Email validation
    cleaned_email = str(email).strip()
    if "@" not in cleaned_email or len(cleaned_email) < 3:
        return False, f"Email '{cleaned_email}' must contain '@'."

    # 4. Grade validation
    cleaned_grade = str(grade).strip().upper()
    if cleaned_grade not in VALID_GRADES:
        return False, f"Grade '{grade}' is invalid. Allowed: {sorted(VALID_GRADES)}"

    parsed_record = {
        "name": cleaned_name,
        "age": age_int,
        "email": cleaned_email,
        "grade": cleaned_grade
    }
    return True, parsed_record

# --- CRUD OPERATIONS ---

def add_student(students):
    """Prompt for student info, validate, and add to list."""
    print("\n--- Add New Student ---")
    name = input("Name: ")
    age = input("Age: ")
    email = input("Email: ")
    grade = input("Grade (A/B/C/D/F): ")

    is_valid, result = validate_student(name, age, email, grade)
    if is_valid:
        students.append(result)
        print(f"Student '{result['name']}' added successfully.")
    else:
        print(f"Validation failed: {result}")

def list_students(students):
    """Display all students in a formatted table."""
    if not students:
        print("\nNo student records found.")
        return

    print("\n" + "=" * 65)
    print(f"{'#':<4} {'Name':<20} {'Age':<6} {'Grade':<8} {'Email'}")
    print("-" * 65)
    for idx, s in enumerate(students, start=1):
        name = s.get("name", "Unknown")[:18]
        age = s.get("age", "-")
        grade = s.get("grade", "-")
        email = s.get("email", "-")
        print(f"{idx:<4} {name:<20} {age:<6} {grade:<8} {email}")
    print("=" * 65)

def search_student(students):
    """Search by name (case-insensitive partial match)."""
    term = input("\nEnter search name/term: ").strip().lower()
    if not term:
        print("Search term cannot be empty.")
        return

    matches = [s for s in students if term in s.get("name", "").lower()]
    if matches:
        print(f"\nFound {len(matches)} matching student(s):")
        list_students(matches)
    else:
        print(f"No students found matching '{term}'.")

def update_student(students):
    """Update a student's grade or email by name."""
    target_name = input("\nEnter the student name to update: ").strip().lower()
    matches = [s for s in students if s.get("name", "").lower() == target_name]

    if not matches:
        print(f"No student found with the exact name '{target_name}'.")
        return

    student = matches[0]
    print(f"Found record: {student['name']} | Age: {student['age']} | Grade: {student['grade']} | Email: {student['email']}")

    field = input("What field would you like to update? [G]rade or [E]mail: ").strip().lower()
    if field == "g":
        new_grade = input("Enter new grade (A/B/C/D/F): ").strip().upper()
        if new_grade in VALID_GRADES:
            student["grade"] = new_grade
            print("Grade updated successfully.")
        else:
            print(f"Invalid grade. Allowed: {sorted(VALID_GRADES)}")
    elif field == "e":
        new_email = input("Enter new email: ").strip()
        if "@" in new_email:
            student["email"] = new_email
            print("Email updated successfully.")
        else:
            print("Invalid email format (must contain '@').")
    else:
        print("Invalid field selection. Choose 'G' or 'E'.")

def delete_student(students):
    """Remove a student by name."""
    target_name = input("\nEnter student name to delete: ").strip().lower()
    for idx, s in enumerate(students):
        if s.get("name", "").lower() == target_name:
            confirm = input(f"Are you sure you want to delete '{s['name']}'? (y/n): ").strip().lower()
            if confirm == "y":
                removed = students.pop(idx)
                print(f"Student '{removed['name']}' removed from database.")
            else:
                print("Deletion cancelled.")
            return

    print(f"Student '{target_name}' not found.")

# --- IMPORT / EXPORT ---

def import_csv(students, csv_filename):
    """Import students from CSV, validating each row."""
    try:
        with open(csv_filename, "r", encoding="utf-8", newline="") as f:
            reader = csv.DictReader(f)
            if not reader.fieldnames:
                print(f"CSV file '{csv_filename}' has no header or is empty.")
                return

            added_count = 0
            skipped_count = 0

            for row_idx, row in enumerate(reader, start=2):
                name = row.get("name")
                age = row.get("age")
                email = row.get("email")
                grade = row.get("grade")

                is_valid, result = validate_student(name, age, email, grade)
                if is_valid:
                    students.append(result)
                    added_count += 1
                else:
                    skipped_count += 1
                    print(f"  • Skipped row {row_idx}: {result} | Raw data: {row}")

            print(f"\nImport finished: {added_count} added, {skipped_count} skipped.")
    except FileNotFoundError:
        print(f"Error: File '{csv_filename}' not found.")

def export_csv(students, csv_filename):
    """Export all students to a CSV file."""
    if not students:
        print("No student records available to export.")
        return

    fieldnames = ["name", "age", "email", "grade"]
    with open(csv_filename, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(students)

    print(f"Exported {len(students)} record(s) to '{csv_filename}'.")

# --- MAIN MENU ---

def run_student_db():
    """Main menu loop."""
    students = load_db(DB_FILE)
    print(f"Loaded {len(students)} student(s) from database.")

    while True:
        print("\n===== STUDENT RECORDS DATABASE =====")
        print("[A]dd  [L]ist  [S]earch  [U]pdate  [D]elete")
        print("[I]mport CSV  [E]xport CSV  [Q]uit")
        choice = input("> ").strip().lower()

        if choice == "a":
            add_student(students)
            save_db(DB_FILE, students)
        elif choice == "l":
            list_students(students)
        elif choice == "s":
            search_student(students)
        elif choice == "u":
            update_student(students)
            save_db(DB_FILE, students)
        elif choice == "d":
            delete_student(students)
            save_db(DB_FILE, students)
        elif choice == "i":
            csv_file = input("CSV filename to import: ").strip()
            import_csv(students, csv_file)
            save_db(DB_FILE, students)
        elif choice == "e":
            csv_file = input("CSV filename to export to: ").strip()
            export_csv(students, csv_file)
        elif choice == "q":
            save_db(DB_FILE, students)
            print("Database saved. Goodbye!")
            break
        else:
            print("Invalid option. Please try again.")

if __name__ == "__main__":
    run_student_db()

Loaded 0 student(s) from database.

===== STUDENT RECORDS DATABASE =====
[A]dd  [L]ist  [S]earch  [U]pdate  [D]elete
[I]mport CSV  [E]xport CSV  [Q]uit

--- Add New Student ---
Student 'Issago' added successfully.

===== STUDENT RECORDS DATABASE =====
[A]dd  [L]ist  [S]earch  [U]pdate  [D]elete
[I]mport CSV  [E]xport CSV  [Q]uit

#    Name                 Age    Grade    Email
-----------------------------------------------------------------
1    Issago               12     A        issago.mosikare@gmail.com

===== STUDENT RECORDS DATABASE =====
[A]dd  [L]ist  [S]earch  [U]pdate  [D]elete
[I]mport CSV  [E]xport CSV  [Q]uit
Database saved. Goodbye!


In [18]:
# --- Generate a sample CSV for testing the Import feature ---

sample_import = """name,age,email,grade
Frank,21,frank@uni.edu,B
Grace,23,grace@uni.edu,A
Hank,not_a_number,hank@uni.edu,C
Ivy,20,ivy_no_at_sign,A
Jack,22,jack@uni.edu,Z
Karen,24,karen@uni.edu,B"""

with open("import_students.csv", "w", newline="") as f:
    f.write(sample_import)

print("import_students.csv created (3 valid, 3 invalid rows).")

import_students.csv created (3 valid, 3 invalid rows).
